In [ ]:
# os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
# os.environ["TF_XLA_FLAGS=--tf_xla_enable_xla_devices"] = "false"
import numpy as np
import tensorflow as tf

from train.models import FullModel
from util import dataset as u_dataset
from util import image as u_image

In [ ]:
def camera_from_label(label):
    """Calculate the camera roll pitch and height from the camera pose in the data.

    Args:
        label: the label with the camera pose

    Returns:
        A tuple of roll, pitch and height.
    """
    alpha = np.arccos(label["cpose"]["z"][2])
    if np.abs(alpha) < 0.01:
        roll = pitch = 0
    else:
        sin_alpha = np.sqrt(1 - label["cpose"]["z"][2] * label["cpose"]["z"][2])
        roll = label["cpose"]["z"][1] / sin_alpha * alpha
        pitch = -label["cpose"]["z"][0] / sin_alpha * alpha
    height = label["cpose"]["h"] * 0.001
    return (roll, pitch, height)

In [ ]:
def intrinsics_from_label(label):
    """
    Get the camera intrinsics from the label.

    Args:
        label: A label from the dataset

    Returns:
        The camera intrinsics as a tuple (cx, cy, fx, fy).
    """

    return (
        label["cintr"]["cx"],
        label["cintr"]["cy"],
        label["cintr"]["fx"],
        label["cintr"]["fy"],
    )

In [ ]:
def get_dataset(directory):
    # Load the dataset
    # TODO: must be divisible by 32
    labels = u_dataset.load_labels(directory)[100:196]
    # labels = u_dataset.load_labels(directory)[:768]

    images = []
    cameras = []
    intrinsics = []
    objectness_mask = []
    offsets = []
    loss_mask = []

    for label in labels:
        # Load the image for the label in YUYV format
        images.append(u_dataset.load_image(directory, label, image_format=u_image.ImageFormat.YUYV))
        # Load the camera pose for the label (roll, pitch, height)
        cameras.append(camera_from_label(label))
        # Load the camera intrinsics for the label
        intrinsics.append(intrinsics_from_label(label))

        masks = u_dataset.get_masks(label, "ball")
        # Load the offsets for the label
        offsets.append(masks[0])
        # Load the objectsness mask for the label
        objectness_mask.append(masks[1])
        # Load the loss mask for the label
        loss_mask.append(masks[2])

    # Combine the images, cameras and intrinsics into a single tensorflow dataset
    ds = tf.data.Dataset.from_tensor_slices(
        {
            "image": tf.reshape(tf.constant(images), (-1, 480, 320, 4)),
            "camera": tf.constant(cameras, dtype=tf.float32),
            "intrinsics": tf.constant(intrinsics, dtype=tf.float32),
            "objectness_mask": tf.reshape(tf.cast(objectness_mask, dtype=tf.float32), (-1, 15, 20)),
            "offsets": tf.reshape(offsets, (-1, 15, 20, 2)),
            "loss_mask": tf.reshape(tf.cast(loss_mask, dtype=tf.float32), (-1, 15, 20)),
        }
    )
    return ds

In [ ]:
test_directory = "data/Joerg_Joerg_CompetitionWalk_GO2025__HULKs_2ndHalf_5"
label = u_dataset.load_labels(test_directory)[165]

offsets, objectness, loss_mask = u_dataset.get_masks(label, "ball")

res_out = tf.shape(offsets)[-3:-1]

scale = tf.cast(((480, 640) / res_out)[::-1], offsets.dtype)
pixels = tf.cast(
    tf.stack(tf.meshgrid(tf.range(res_out[1]), tf.range(res_out[0])), axis=-1),
    offsets.dtype,
)
coords = tf.reshape((offsets + pixels - 0.5) * scale, (1, 20 * 15, 2))

# print(coords)
print(offsets)

u_image.show_cell_on_image(test_directory, label, "ball", grid_dims=(15, 20))

In [ ]:
train_ds = get_dataset("data/Joerg_Joerg_CompetitionWalk_GO2025__HULKs_2ndHalf_5")
train_ds = train_ds.shuffle(32, seed=42)
train_ds = train_ds.batch(32)
# print(train_ds)

In [ ]:
# Upper camera dimensions. Width is halved because of YUYV format
model = FullModel(480, 320)
model.compile(optimizer=tf.keras.optimizers.Adam())
model.fit(x=train_ds, epochs=200)

# print(model.summary())


# Save model
# model.save("./models/model.keras")

In [ ]:
for element in train_ds:
    print(element.keys())
    print(element["image"].shape)
    print(element["camera"].shape)
    print(element["intrinsics"].shape)
    print(element["objectness_mask"].shape)
    print(element["offsets"].shape)
    break

In [ ]:
test_directory = "data/Joerg_Joerg_CompetitionWalk_GO2025__HULKs_2ndHalf_5"

label = u_dataset.load_labels(test_directory)[160]

image = u_dataset.load_image(test_directory, label, image_format=u_image.ImageFormat.YUYV)
camera = camera_from_label(label)
intrinsics = intrinsics_from_label(label)

image_yuv = tf.reshape(tf.constant(image), (-1, 480, 320, 4))
camera = tf.constant(camera, dtype=tf.float32)
intrinsics = tf.constant(intrinsics, dtype=tf.float32)

print("Image: ", image_yuv.shape)
print("Camera: ", camera)
print("Intrinsics: ", intrinsics)


results = model((image_yuv, camera, intrinsics), training=None)


In [ ]:
# print(results["ball"][1])

# camera_rays, camera_rotation, rotated_camera_rays, camera_height, factors, distances_in_camera, positions_in_camera, pixel_sizes

camera_rays = results["ball"][3]
camera_rotation = results["ball"][4]
rotated_camera_rays = results["ball"][5]
camera_height = results["ball"][6]
factors = results["ball"][7]
distances_in_camera = results["ball"][8]
positions_in_camera = results["ball"][9]
pixel_sizes = results["ball"][10]
coords_results = results["ball"][11]
intrinsics_results = results["ball"][12]
masks = results["ball"][1]

# print("Intrinsics: ", intrinsics_results)
print("Factors: ", factors)
# print("Distances: ", distances_in_camera)
print("Camera rays: ", camera_rays)
print("Rotated Camera Rays: ", rotated_camera_rays)
print("Positions: ", positions_in_camera)
# print("Camera Height: ", camera_height)
print("Camera Rotation: ", camera_rotation)
# print("Coords: ", coords_results)
print("Masks: ", masks)


u_image.show_patches_on_image(image, "ball", results)

In [ ]:
print(model.encoder.output_shape)
print(model.encoder.input_shape)

print(model.categories["ball"])